# Meal merging and source-data alignment

This notebook combines individual food records into meals and aligns dietary records with participant metadata and continuous glucose monitoring (CGM) coverage. The merged meal table is the input to preprocessing notebook 02.

## Notebook flow

1. Load participant metadata, CGM measurements and dietary records, and align participant keys and timestamps.
2. Restrict food records to each participant's CGM observation period, remove listed water entries, and exclude food entries weighing 2 kg or more.
3. Merge food entries within 30-minute windows for each participant, summing nutrients and retaining food identifiers and descriptions.
4. Assign meal-time categories and export the merged table.

## Inputs

Source files in `Data/raw data/`: `metadata.csv`, `cgm_data.csv` and `mfr_food_and_you.csv`. These source files are required to run the notebook and are not distributed in this repository. Paths are defined in `Code/data_paths.py`.

## Outputs

| File | Contents |
| --- | --- |
| `Data/raw data/meal_data.csv` | Merged meal records with nutrient totals, food identifiers and descriptions, participant keys, timestamps and meal-time categories. |

Dataset previews, participant counts and filtering summaries are displayed in the notebook.

## 1. Setup

Resolve project paths and import the data-processing packages.

In [93]:
from pathlib import Path
import sys

# Find the project when launched from its root, Code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "Code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
from data_paths import (
    RAW_DATA_DIR, RAW_METADATA_PATH, RAW_CGM_PATH, RAW_FOOD_PATH,
    MERGED_MEALS_PATH, CLEANED_FOOD_PATH,
)

import pandas as pd
import numpy as np

from tqdm import tqdm
from datetime import datetime, timedelta

pd.set_option('mode.chained_assignment', None)

## 2. Load and align source records

Read participant metadata, CGM measurements and dietary records, then align participant keys and timestamps.

In [96]:
meta_data = pd.read_csv(RAW_METADATA_PATH).reset_index(drop=True)
meta_data.rename(columns={"subject_app_key": "subject_key"}, inplace=True)
id_to_subjectKey = dict(zip(meta_data["id"], meta_data["subject_key"]))

meta_data["bmi"] = meta_data["weight"] / ((meta_data["height"] / 100) ** 2)
meta_data["WaistHipRatio"] = meta_data["waist"] / meta_data["hip"]
meta_data["WaistHeightRatio"] = meta_data["waist"] / meta_data["height"]
meta_data

,subject_key,id,gender,waist,hip,weight,height,cohort,age,bmi,WaistHipRatio,WaistHeightRatio
0,ce3m4w,1939,female,65.0,91.2,42.0,150.0,cohort_c,20,18.666667,0.712719,0.433333
1,qrh38g,2509,male,103.0,104.1,90.0,175.0,cohort_b,43,29.387755,0.989433,0.588571
2,kz2eh8,1140,male,79.0,88.5,61.0,175.0,cohort_b,33,19.918367,0.892655,0.451429
3,curus8,2544,male,79.6,95.3,66.0,181.0,cohort_b,29,20.145905,0.835257,0.439779
4,we2j7w,2261,female,87.0,114.0,73.0,164.0,cohort_b,41,27.141582,0.763158,0.530488
...,...,...,...,...,...,...,...,...,...,...,...,...
1005,qda49t,2655,male,101.3,103.0,86.0,173.0,cohort_b,55,28.734672,0.983495,0.585549
1006,phzbkm,2517,male,102.1,104.5,84.0,170.0,cohort_b,30,29.065744,0.977033,0.600588
1007,mbrack,2507,female,71.3,95.2,56.0,173.0,cohort_b,26,18.710949,0.748950,0.412139
1008,m93aw9,2653,male,86.0,99.0,81.0,191.0,cohort_b,34,22.203339,0.868687,0.450262


In [97]:
### glucose data 
df_gluc = pd.read_csv(RAW_CGM_PATH, index_col=False).reset_index(drop=True)

df_gluc.rename(columns={'read_at':'time', 'user_id':'id', 'val':'gl_mmol'}, inplace=True)

df_gluc['subject_key'] = df_gluc['id'].map(id_to_subjectKey)
df_gluc = df_gluc[~df_gluc['subject_key'].isna()]

df_gluc['time'] = pd.to_datetime(df_gluc['time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
df_gluc = df_gluc.dropna(subset=['time'])
df_gluc.sort_values(by=['subject_key', 'time'], inplace=True)

print(df_gluc.shape)
df_gluc.head()

(1522406, 4)


,time,id,gl_mmol,subject_key
0,2018-11-26 08:00:00,5,6.06,02ae3856ca04
1,2018-11-26 08:15:00,5,7.55,02ae3856ca04
2,2018-11-26 08:30:00,5,8.51,02ae3856ca04
3,2018-11-26 08:45:00,5,7.39,02ae3856ca04
4,2018-11-26 09:00:00,5,6.80,02ae3856ca04


In [101]:
### food data 
df_food = pd.read_csv(RAW_FOOD_PATH, index_col=0, low_memory=False)

# Parse timestamps without a timezone suffix.
df_food['eaten_at'] = df_food["eaten_at"].apply(
    lambda x: datetime.strptime(x, "%Y-%m-%d %H:%M:%S") if pd.notna(x) else pd.NaT)

df_food['local_eaten_at'] = df_food["local_eaten_at"].apply(
    lambda x: datetime.strptime(x, "%Y-%m-%d %H:%M:%S") if pd.notna(x) else pd.NaT)

df_food.reset_index(inplace=True)

df_food = df_food[df_food['subject_key'].isin(df_gluc['subject_key'].unique())]

df_food['combined_name'] = df_food['display_name_en'].fillna(
                                df_food['display_name_de']).fillna(
                                df_food['display_name_fr']).fillna("unknown")

print(df_food.shape)
df_food.head()

(511518, 97)


,food_id,barcode,dish_id,eaten_quantity,eaten_unit,subject_key,eaten_at,eaten_at_utc_offset,media_count,type,...,vitamin_c_eaten,vitamin_d_eaten,zinc_eaten,dairy_products_meat_fish_eggs_tofu,vegetables_fruits,sweets_salty_snacks_alcohol,non_alcoholic_beverages,grains_potatoes_pulses,oils_fats_nuts,composite-foods
0,13,NaN,335994,110.0,g,bjsqab,2022-02-15 18:06:27,60,1,generic,...,0.00000,0.0,0.004730,110.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13,NaN,333896,150.0,g,dqbtrp,2022-02-12 10:42:09,60,1,generic,...,0.00000,0.0,0.006450,150.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13,NaN,380245,65.0,g,gb2gmh,2022-08-26 18:32:39,120,1,generic,...,0.00000,0.0,0.002795,65.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13,NaN,411384,100.0,g,erv9jd,2023-01-18 13:08:53,60,1,generic,...,0.00000,0.0,0.004300,100.0,0.0,0.0,0.0,0.0,0.0,0.0
4,25,NaN,202497,35.0,g,6ata8r,2021-03-30 13:52:23,120,1,generic,...,0.00315,0.0,0.000035,0.0,35.0,0.0,0.0,0.0,0.0,0.0


In [102]:
df_food.columns

Index(['food_id', 'barcode', 'dish_id', 'eaten_quantity', 'eaten_unit',
       'subject_key', 'eaten_at', 'eaten_at_utc_offset', 'media_count', 'type',
       'display_name_en', 'display_name_fr', 'display_name_de',
       'fallback_food_id', 'standard_portion_quantity',
       'standard_portion_unit', 'specific_gravity', 'alcohol',
       'all_trans_retinol_equivalents_activity', 'beta_carotene',
       'beta_carotene_activity', 'calcium', 'carbohydrates', 'chloride',
       'cholesterol', 'energy_kcal', 'energy_kj', 'fat',
       'fatty_acids_monounsaturated', 'fatty_acids_polyunsaturated',
       'fatty_acids_saturated', 'fiber', 'folate', 'iodide', 'iron',
       'magnesium', 'niacin', 'pantothenic_acid', 'phosphorus', 'potassium',
       'protein', 'salt', 'sodium', 'starch', 'sugar', 'vitamin_a_activity',
       'vitamin_b1', 'vitamin_b12', 'vitamin_b2', 'vitamin_b6', 'vitamin_c',
       'vitamin_d', 'vitamin_e_activity', 'water', 'zinc',
       'eaten_quantity_in_gram', 'energy_

In [103]:
# Convert to sets
set1 = set(meta_data["subject_key"].unique())
set2 = set(df_food["subject_key"].unique())
set3 = set(df_gluc["subject_key"].unique())

# Find intersection of keys present in all three
common_keys = set1.intersection(set2, set3)
count_common = len(common_keys)

print("Count of participants present in the three datasets (cgm, food and demographic data) keys:", count_common)

Count of participants present in the three datasets (cgm, food and demographic data) keys: 1010


## 3. Restrict dietary records to CGM coverage

Trim food records to the observed CGM period and remove listed water entries.

In [104]:
# keep food data logged at times [first gluc + offset , last gluc - offset]

def users_food_trimming(df_food, df_gluc, first_meal_gluc_time_diff=4, last_meal_gluc_time_diff=2):
    df_food_trimmed = pd.DataFrame()

    for sk in tqdm(df_food['subject_key'].unique()):
        user_food = df_food[df_food['subject_key']==sk].sort_values(by='eaten_at')
        user_gluc = df_gluc[df_gluc['subject_key']==sk].sort_values(by='time')
        
        user_food_trimmed = user_food[(user_food['eaten_at'] >= (user_gluc['time'].iloc[0] + 
                                        pd.Timedelta(hours=first_meal_gluc_time_diff))) & 
                                    (user_food['eaten_at'] <= user_gluc['time'].iloc[-1] - 
                                        pd.Timedelta(hours=last_meal_gluc_time_diff))
                                    ]

        df_food_trimmed = pd.concat([df_food_trimmed, user_food_trimmed], axis=0)

    return df_food_trimmed

df_food_trimmed = users_food_trimming(df_food, df_gluc, first_meal_gluc_time_diff=0, last_meal_gluc_time_diff=0)
print("After trim:",df_food_trimmed.shape, "\tBefore trim:", df_food.shape)
df_food_trimmed = df_food_trimmed.reset_index()

100%|██████████| 1010/1010 [02:31<00:00,  6.66it/s]


After trim: (412771, 97) 	Before trim: (511518, 97)


In [105]:
print("Original number of standardized meals:")
df_food_trimmed[df_food_trimmed['food_id'].isin([3240])].shape

Original number of standardized meals:


(6878, 98)

In [106]:
# we remove water intakes
water_ids = [2576, 2577, 2578, 2579, 2580, 2581, 31476,
           ## added items
    4134 , ##Vittel Natürliches Mineralwasser 	0.0
	3536 , #Evian - NATÜRLICHES MINERALWASSER 	0.0
	3930 , #Evian - Sport 	0.0
	3472 , #Henniez - Leicht prickelnd 	0.0
	9271 , #Henniez - Prickelnd 	0.0
	3831 , #Valser - Natürliches Mineralwasser ohne Kohlen... 	0.0
	6388 , #Coop Swiss Alpina Quality & Prices 	0.0
 	3537 , #Coop Qualité & Prix - Swiss Alpina, Mineralwas... 	0.0
 	3989 , #Henniez - Ohne Kohlensäure 	0.0
 	6135 , #Valser - Sprudelwasser 	0.0
 	6381 , #San Pellegrino 	0.0
 	5454 , #Valser - Classic 	0.0
 	3603 , #evian Natural Mineral Water 	0.0
 	3749 , #Henniez - Leicht prickelnd 	0.0
 	3819 , #Valser - Silence, still und sanft 	0.0
 	3580 , #Aproz classic - Natürliche Mineralwasser aus d... 	0.0
 	3420 , #Henniez - Eau minérale Légère 	0.0
 	10953 , #Henniez - Wasser ohne Kohlensäure 	0.0
 	13973 , #Aproz - Classic (mit Kohlensäure) 	0.0
 	8997 , #MBudget - Natürliches Mineralwasser mit Kohlen... 	0.0
 	3122 , #Evian - NATÜRLICHES MINERALWASSER 	0.0
 	4267 , #Aquella - Mit Kohlensäure 	0.0
 	8042 , #Cristalp, natural mineral water 	0.0
 	3719 , #Badoit - Leicht Kohlensäurehaltig 	0.0
            ]

## Remove water
remove_water = True

if remove_water:
    df_food_trimmed = df_food_trimmed[~df_food_trimmed['food_id'].isin(water_ids)]
    print("After water removal:",df_food_trimmed.shape, "\tBefore trim:", df_food.shape)

print(df_food_trimmed.shape, df_food.shape)

df_food_trimmed[~(df_food_trimmed['food_id'].isin(water_ids) ) & 
                (df_food_trimmed['eaten_unit'] != 'g')][['combined_name', 'eaten_quantity', 
                                                         'eaten_quantity_in_gram', 'eaten_unit', 'carb_eaten']
                ].sort_values(by='eaten_quantity_in_gram', ascending=False)

After water removal: (354325, 98) 	Before trim: (511518, 97)
(354325, 98) (511518, 97)


,combined_name,eaten_quantity,eaten_quantity_in_gram,eaten_unit,carb_eaten
120859,"Tea, green",438.0,43800.0,dl,0.00
97595,Beer n.s.,3000.0,3000.0,ml,72.00
195277,Beer n.s.,2000.0,2000.0,ml,48.00
146671,Water with lemon juice,2000.0,2000.0,ml,2.00
161311,Beer n.s.,20.0,2000.0,dl,48.00
...,...,...,...,...,...
158831,Milk drink,-1.0,-100.0,dl,-4.70
322614,Milk,-1.0,-100.0,dl,-4.70
158113,Milk,-1.4,-140.0,dl,-6.58
187303,Milk,-1.5,-150.0,dl,-7.05


## 4. Merge food entries into meals

Exclude food entries weighing 2 kg or more, then combine records within 30-minute windows for each participant.

In [107]:
### Merging meals
summable_features = ['eaten_quantity_in_gram','water', 'dairy_products_meat_fish_eggs_tofu', 'vegetables_fruits',
       'sweets_salty_snacks_alcohol', 'non_alcoholic_beverages', 'grains_potatoes_pulses', 'oils_fats_nuts', "composite-foods"]
summable_features += [i for i in df_food.columns if "_eaten" in i if i not in ["local_eaten_at", 'energy_kj_eaten']]

concat_features = ["food_id", "dish_id", "combined_name",'eaten_unit']
static_features = ["subject_key", "eaten_at", "local_eaten_at"]

In [108]:
# we remove all single food intake > 2000g
df_food_trimmed = df_food_trimmed[df_food_trimmed["eaten_quantity_in_gram"]<2000].reset_index(drop=True)

In [109]:
def merge_meals_by_30min(df, summable_features, static_features, concat_features,
                         time_window='30 min', limit=None):
    if df.empty:
        return pd.DataFrame(columns=static_features + concat_features + summable_features)

    df = df.sort_values('eaten_at').reset_index(drop=True)
    tol = pd.Timedelta(time_window)
    results = []

    i = 0
    n = len(df) if limit is None else min(limit, len(df))
    while i < n:
        start = df.loc[i, 'eaten_at']
        mask = (df['eaten_at'] >= start) & (df['eaten_at'] < start + tol)
        sdf = df.loc[mask]

        row = sdf[summable_features].sum()
        for c in static_features:
            row[c] = sdf.iloc[0][c]
        for c in concat_features:
            row[c] = sdf[c].unique().tolist()
        results.append(row)

        i += len(sdf)  # Skip the merged records.

    return pd.DataFrame(results)[static_features + concat_features + summable_features]

In [110]:
merged_foods_users = pd.DataFrame()
df_food_trimmed["eaten_at"]= pd.to_datetime(df_food_trimmed["eaten_at"])

for user in tqdm(df_food_trimmed['subject_key'].unique()):
    user_food_trimmed = df_food_trimmed[df_food_trimmed['subject_key']==user].sort_values(by='eaten_at')

    user_merged = merge_meals_by_30min(user_food_trimmed, summable_features, static_features, concat_features, 
                                  time_window='30 minute', limit=None)
    merged_foods_users = pd.concat([merged_foods_users, user_merged], axis=0)

merged_foods_users.sort_values(by=['subject_key','eaten_at'], inplace=True)
print(merged_foods_users.shape, df_food_trimmed.shape)

100%|██████████| 1002/1002 [06:23<00:00,  2.62it/s]

(105769, 45) (354299, 98)


In [29]:
 # (105769, 45) (354299, 98)

## 5. Assign meal-time categories and export

Label meal times, save the merged meal table and inspect participant meal counts.

In [111]:
## meal type
def get_mealtime_type(time):
    if time.hour < 10 and time.hour > 4:
        return 'breakfast'
    elif time.hour < 16:
        return 'lunch'
    else:
        return 'dinner'

merged_foods_users['mealtime_type'] = merged_foods_users['eaten_at'].apply(get_mealtime_type)

In [113]:
merged_foods_users = merged_foods_users.sort_values(['subject_key', 'eaten_at'])

merged_foods_users.to_csv(MERGED_MEALS_PATH)

In [114]:
merged_foods_users["subject_key"].value_counts()

subject_key
ym5jff    394
epm6fa    282
raz8kc    276
8ncyap    273
r7de9w    266
         ... 
84xjbz     21
gbcxj4     18
c5h78q     16
rsgqwt     13
wpcyhn      8
Name: count, Length: 1002, dtype: int64

-------